In [2]:
import pandas as pd

CSV_PATH = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"

df = pd.read_csv(CSV_PATH)

# คอลัมน์ label
label_col = "Finding Labels"

# แตก multi-label ด้วยตัวคั่น '|'
labels_series = (
    df[label_col]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.split("|")
    .explode()
    .str.strip()
)

# (ถ้ามีค่าว่าง) ตัดทิ้ง
labels_series = labels_series[labels_series != ""]

# นับจำนวนต่อคลาส
class_counts = labels_series.value_counts()

print("=== จำนวนคลาสทั้งหมด (unique) ===")
print(class_counts.shape[0])

print("\n=== Top 20 คลาสที่พบมากสุด ===")
print(class_counts.head(20))

print("\n=== จำนวนตัวอย่างต่อคลาส (ทั้งหมด) ===")
print(class_counts)


=== จำนวนคลาสทั้งหมด (unique) ===
15

=== Top 20 คลาสที่พบมากสุด ===
Finding Labels
No Finding            60361
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                  227
Name: count, dtype: int64

=== จำนวนตัวอย่างต่อคลาส (ทั้งหมด) ===
Finding Labels
No Finding            60361
Infiltration          19894
Effusion              13317
Atelectasis           11559
Nodule                 6331
Mass                   5782
Pneumothorax           5302
Consolidation          4667
Pleural_Thickening     3385
Cardiomegaly           2776
Emphysema              2516
Edema                  2303
Fibrosis               1686
Pneumonia              1431
Hernia                 

In [3]:
# ===== เฉพาะ 5 คลาสที่ต้องการ =====
target_classes = [
    "Atelectasis",
    "Cardiomegaly",
    "Edema",
    "Emphysema",
    "Fibrosis"
]

print("\n=== จำนวนตัวอย่างเฉพาะ 5 คลาสที่กำหนด ===")

for cls in target_classes:
    count = class_counts.get(cls, 0)
    print(f"{cls}: {count}")



=== จำนวนตัวอย่างเฉพาะ 5 คลาสที่กำหนด ===
Atelectasis: 11559
Cardiomegaly: 2776
Edema: 2303
Emphysema: 2516
Fibrosis: 1686


In [3]:
total_images = len(df)

selected_df = pd.DataFrame({
    "Count": class_counts.reindex(target_classes).fillna(0).astype(int)
})

selected_df["Percentage (%)"] = (selected_df["Count"] / total_images * 100).round(2)

print("\n=== Count + Percentage ===")
print(selected_df)



=== Count + Percentage ===
                Count  Percentage (%)
Finding Labels                       
Atelectasis     11559           10.31
Cardiomegaly     2776            2.48
Edema            2303            2.05
Emphysema        2516            2.24
Fibrosis         1686            1.50


คำนวณ weight balance 5 class

In [4]:
import numpy as np

counts = np.array([11559, 2776, 2303, 2516, 1686], dtype=np.float32)
N = counts.sum()

w = np.sqrt(N / counts)                 # sqrt inverse freq
w = w / w.mean()                        # normalize mean=1

classes = ["Atelectasis", "Cardiomegaly", "Edema", "Emphysema", "Fibrosis"]
for c, wi in zip(classes, w):
    print(f"{c:15s} weight = {wi:.3f}")


Atelectasis     weight = 0.498
Cardiomegaly    weight = 1.016
Edema           weight = 1.115
Emphysema       weight = 1.067
Fibrosis        weight = 1.304


ใช้ tensoflow

In [1]:
import tensorflow as tf

# ใส่ weights ที่คำนวณได้ (5 ค่า)
w = tf.constant([1.0, 1.0, 1.0, 1.0, 1.0], dtype=tf.float32)  # <-- replace

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True, reduction="none")

def weighted_bce(y_true, y_pred):
    # y_true,y_pred shape: (batch, 5)
    loss = bce(y_true, y_pred)  # (batch,)
    # ทำ weight แบบ per-class: ต้องคำนวณ element-wise
    per_elem = tf.nn.sigmoid_cross_entropy_with_logits(labels=y_true, logits=y_pred)  # (batch,5)
    weighted = per_elem * w  # broadcast
    return tf.reduce_mean(tf.reduce_sum(weighted, axis=-1))


In [5]:
import numpy as np

counts = np.array([11559, 2776, 2303, 2516, 1686], dtype=np.float32)
N = counts.sum()

w = np.sqrt(N / counts)                 # sqrt inverse freq
w = w / w.mean()                        # normalize mean=1

classes = ["Atelectasis", "Cardiomegaly", "Edema", "Emphysema", "Fibrosis"]
for c, wi in zip(classes, w):
    print(f"{c:15s} weight = {wi:.3f}")


Atelectasis     weight = 0.498
Cardiomegaly    weight = 1.016
Edema           weight = 1.115
Emphysema       weight = 1.067
Fibrosis        weight = 1.304


In [1]:
import os
import random
import pandas as pd
from collections import Counter

random.seed(42)

CSV_PATH = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"
TRAIN_LIST = r"D:\senior-project\ploy-senior-project\archive\train_val_list.txt"
OUT_LIST = r"D:\senior-project\ploy-senior-project\archive\train_val_list_oversampled.txt"

TARGET_CLASSES = ["Atelectasis","Cardiomegaly","Edema","Emphysema","Fibrosis"]

# เลือก target ต่อคลาส (ปรับได้)
# 1) ใกล้เคียงสุด: เท่าคลาสใหญ่สุด (11559)
# TARGET_PER_CLASS = 11559

# 2) แนะนำลดซ้ำลง: เอาประมาณ 6000
TARGET_PER_CLASS = 6000

# -------------------------
# 1) โหลด CSV แล้วทำ multi-hot สำหรับ 5 คลาส
df = pd.read_csv(CSV_PATH)
y = df["Finding Labels"].fillna("").astype(str).str.get_dummies(sep="|")
y = y.rename(columns=lambda c: c.strip())
y5 = y.reindex(columns=TARGET_CLASSES, fill_value=0)

# สร้าง map: image -> labels (0/1)
img_to_vec = dict(zip(df["Image Index"].astype(str), y5.values))

# -------------------------
# 2) อ่าน train list แล้วดึงแค่ชื่อไฟล์ออกมา
with open(TRAIN_LIST, "r", encoding="utf-8") as f:
    lines = [ln.strip() for ln in f if ln.strip()]

# สมมติว่า train_val_list.txt แต่ละบรรทัดมี path หรือชื่อไฟล์
# เราจะใช้ basename เป็น key เทียบกับ CSV
def get_img_name(line: str) -> str:
    return os.path.basename(line.split()[0])  # รองรับกรณีมี label ต่อท้าย

train_items = []
for ln in lines:
    img = get_img_name(ln)
    if img in img_to_vec:
        train_items.append(ln)

print("Train items (matched with CSV):", len(train_items))

# -------------------------
# 3) ฟังก์ชันนับจำนวน positive ต่อคลาสจาก list ปัจจุบัน
def count_per_class(items):
    counts = Counter({c:0 for c in TARGET_CLASSES})
    for ln in items:
        img = get_img_name(ln)
        vec = img_to_vec.get(img, None)
        if vec is None: 
            continue
        for i, c in enumerate(TARGET_CLASSES):
            counts[c] += int(vec[i])
    return counts

base_counts = count_per_class(train_items)
print("\n=== Base counts in TRAIN ===")
for c in TARGET_CLASSES:
    print(f"{c:15s}: {base_counts[c]}")

# -------------------------
# 4) เตรียม pool ของรูปสำหรับแต่ละคลาส (เอาเฉพาะรูปที่มีคลาสนั้น)
pool = {c: [] for c in TARGET_CLASSES}
for ln in train_items:
    img = get_img_name(ln)
    vec = img_to_vec[img]
    for i, c in enumerate(TARGET_CLASSES):
        if vec[i] == 1:
            pool[c].append(ln)

for c in TARGET_CLASSES:
    print(f"Pool[{c}] size:", len(pool[c]))

# -------------------------
# 5) Oversample แบบ greedy: เติมคลาสที่ขาดให้ถึง TARGET_PER_CLASS
new_items = list(train_items)
cur_counts = Counter(base_counts)

MAX_ITERS = 200000  # กันลูปยาวเกิน
iters = 0

def deficits():
    return {c: max(0, TARGET_PER_CLASS - cur_counts[c]) for c in TARGET_CLASSES}

while True:
    iters += 1
    d = deficits()
    # ถ้าทุกคลาสถึงแล้ว ออก
    if all(v == 0 for v in d.values()):
        break
    if iters > MAX_ITERS:
        print("Reached MAX_ITERS, stop.")
        break

    # เลือกคลาสที่ขาดมากสุดก่อน
    c_star = max(d, key=d.get)
    if d[c_star] == 0:
        break

    # ถ้า pool ว่าง แปลว่า train ไม่มีคลาสนี้เลย (กรณีแปลก) -> ข้าม
    if not pool[c_star]:
        print("No samples in train pool for:", c_star)
        cur_counts[c_star] = TARGET_PER_CLASS
        continue

    # สุ่มเพิ่ม 1 sample จาก pool ของคลาสนั้น
    chosen = random.choice(pool[c_star])
    new_items.append(chosen)

    # อัปเดต counts (สำคัญ: 1 รูปอาจเพิ่มหลายคลาสพร้อมกัน)
    img = get_img_name(chosen)
    vec = img_to_vec[img]
    for i, c in enumerate(TARGET_CLASSES):
        cur_counts[c] += int(vec[i])

# -------------------------
# 6) เขียนไฟล์ train ใหม่
with open(OUT_LIST, "w", encoding="utf-8") as f:
    for ln in new_items:
        f.write(ln + "\n")

print("\n✅ Wrote:", OUT_LIST)
print("Original train size:", len(train_items))
print("New train size:", len(new_items))

print("\n=== New counts in TRAIN (after oversample) ===")
for c in TARGET_CLASSES:
    print(f"{c:15s}: {cur_counts[c]}")


Train items (matched with CSV): 86524

=== Base counts in TRAIN ===
Atelectasis    : 8280
Cardiomegaly   : 1707
Edema          : 1378
Emphysema      : 1423
Fibrosis       : 1251
Pool[Atelectasis] size: 8280
Pool[Cardiomegaly] size: 1707
Pool[Edema] size: 1378
Pool[Emphysema] size: 1423
Pool[Fibrosis] size: 1251

✅ Wrote: D:\senior-project\ploy-senior-project\archive\train_val_list_oversampled.txt
Original train size: 86524
New train size: 103866

=== New counts in TRAIN (after oversample) ===
Atelectasis    : 10721
Cardiomegaly   : 6000
Edema          : 6000
Emphysema      : 6000
Fibrosis       : 6000


In [5]:
import os

base_path = "archive/images_001"

df["path"] = df["Image Index"].apply(lambda x: os.path.join(base_path, x))


In [4]:
from pathlib import Path

img_root = Path("../archive/images")

all_imgs = {p.name: str(p) for p in img_root.rglob("*.png")}

df["path"] = df["Image Index"].map(all_imgs.get)
df = df[df["path"].notna()].reset_index(drop=True)

print("Images matched:", len(df))


Images matched: 0


In [1]:
import pandas as pd
from collections import Counter

# =========================
# CONFIG
# =========================
CSV_PATH = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"

# =========================
# LOAD CSV
# =========================
df = pd.read_csv(CSV_PATH)

# =========================
# EXTRACT LABELS
# =========================
all_labels = []

for labels in df["Finding Labels"].dropna():

    # split multi-label
    split_labels = labels.split("|")

    for label in split_labels:

        # clean text
        label = label.strip()

        # skip empty
        if label == "":
            continue

        all_labels.append(label)

# =========================
# COUNT LABELS
# =========================
label_counter = Counter(all_labels)

# =========================
# SORT
# =========================
sorted_labels = sorted(label_counter.items(), key=lambda x: x[1], reverse=True)

# =========================
# SHOW RESULT
# =========================
print("=" * 80)
print(f"จำนวนโรค/label ไม่ซ้ำทั้งหมด: {len(sorted_labels)}")
print("=" * 80)

for i, (label, count) in enumerate(sorted_labels, 1):
    print(f"{i}. {label} : {count}")

จำนวนโรค/label ไม่ซ้ำทั้งหมด: 15
1. No Finding : 60361
2. Infiltration : 19894
3. Effusion : 13317
4. Atelectasis : 11559
5. Nodule : 6331
6. Mass : 5782
7. Pneumothorax : 5302
8. Consolidation : 4667
9. Pleural_Thickening : 3385
10. Cardiomegaly : 2776
11. Emphysema : 2516
12. Edema : 2303
13. Fibrosis : 1686
14. Pneumonia : 1431
15. Hernia : 227


In [1]:
import pandas as pd
import ast
from collections import Counter

# =========================
# PATH
# =========================
NIH_CSV = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"

PAD_CSV = PAD_CSV = r"D:\senior-project\ploy-senior-project\padchest_AP_chest\PADCHEST_chest_x_ray_images_labels.csv"

# =========================
# NIH COUNT
# =========================
nih_df = pd.read_csv(NIH_CSV)

nih_counter = Counter()

for labels in nih_df["Finding Labels"].dropna():

    split_labels = labels.split("|")

    for label in split_labels:

        label = label.strip()

        if label != "":
            nih_counter[label] += 1

# =========================
# PADCHEST MAPPING
# =========================
mapping = {
    "Atelectasis": ["atelectasis"],
    "Cardiomegaly": ["cardiomegaly"],
    "Consolidation": ["consolidation"],
    "Effusion": ["effusion", "pleural effusion"],
    "Emphysema": ["emphysema"],
    "Fibrosis": ["fibrosis"],
    "Hernia": ["hernia"],
    "Infiltration": ["infiltrates", "interstitial pattern"],
    "Mass": ["mass"],
    "No Finding": ["normal"],
    "Nodule": ["nodule"],
    "Pleural_Thickening": ["pleural thickening"],
    "Pneumonia": ["pneumonia"],
    "Pneumothorax": ["pneumothorax"],
    "Edema": ["edema"]
}

# =========================
# LOAD PADCHEST
# =========================
pad_df = pd.read_csv(PAD_CSV)

# ใช้ AP เท่านั้น
pad_df = pad_df[
    pad_df["Projection"].astype(str).str.strip() == "AP"
].copy()

# =========================
# CLEAN LABELS
# =========================
def clean_labels(text):

    if pd.isna(text):
        return []

    try:
        parsed = ast.literal_eval(text)

        return [
            str(x).lower().strip()
            for x in parsed
        ]

    except:
        return []

# =========================
# COUNT PADCHEST
# =========================
pad_counter = Counter()

for _, row in pad_df.iterrows():

    labels = clean_labels(row["Labels"])

    for disease, keywords in mapping.items():

        if any(
            keyword in labels
            for keyword in keywords
        ):
            pad_counter[disease] += 1

# =========================
# FINAL TABLE
# =========================
rows = []

all_diseases = sorted(mapping.keys())

for disease in all_diseases:

    nih_count = nih_counter.get(disease, 0)
    pad_count = pad_counter.get(disease, 0)

    rows.append({
        "Disease": disease,
        "NIH_Count": nih_count,
        "PadChest_AP_Count": pad_count,
        "Hybrid_Total": nih_count + pad_count
    })

summary_df = pd.DataFrame(rows)

# =========================
# SHOW
# =========================
print("=" * 100)
print("NIH vs PadChest AP Comparison")
print("=" * 100)

print(summary_df.to_string(index=False))

# save
summary_df.to_csv(
    "nih_vs_padchest_comparison.csv",
    index=False
)

print("\nบันทึกไฟล์ nih_vs_padchest_comparison.csv แล้ว")

NIH vs PadChest AP Comparison
           Disease  NIH_Count  PadChest_AP_Count  Hybrid_Total
       Atelectasis      11559                286         11845
      Cardiomegaly       2776                359          3135
     Consolidation       4667                140          4807
             Edema       2303                  0          2303
          Effusion      13317                616         13933
         Emphysema       2516                 24          2540
          Fibrosis       1686                  0          1686
            Hernia        227                  0           227
      Infiltration      19894                972         20866
              Mass       5782                  1          5783
        No Finding      60361                633         60994
            Nodule       6331                 81          6412
Pleural_Thickening       3385                 16          3401
         Pneumonia       1431                475          1906
      Pneumothorax       

In [2]:
import os
import pandas as pd

IMG_DIR = r"D:\senior-project\ploy-senior-project\padchest_AP_chest"
OUTPUT_CSV = r"D:\senior-project\ploy-senior-project\new-train\padchest_train_from_folder.csv"

CLASSES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]

rows = []

for disease in CLASSES:
    disease_folder = os.path.join(IMG_DIR, disease)

    if not os.path.exists(disease_folder):
        print("ไม่พบ folder:", disease_folder)
        continue

    for filename in os.listdir(disease_folder):
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            row = {"Filename": f"{disease}/{filename}"}

            for cls in CLASSES:
                row[cls] = 1 if cls == disease else 0

            rows.append(row)

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print("สร้าง CSV สำเร็จ:", OUTPUT_CSV)
print("จำนวนรูปทั้งหมด:", len(df))
print(df.head())

ไม่พบ folder: D:\senior-project\ploy-senior-project\padchest_AP_chest\Emphysema
สร้าง CSV สำเร็จ: D:\senior-project\ploy-senior-project\new-train\padchest_train_from_folder.csv
จำนวนรูปทั้งหมด: 1448
                                            Filename  Atelectasis  \
0  Atelectasis/1106578233752450596532946345616991...            1   
1   Atelectasis/127522431331980803496_00-069-065.png            1   
2   Atelectasis/127522431331980859708_00-095-174.png            1   
3   Atelectasis/127522431331980861747_00-095-108.png            1   
4  Atelectasis/1275224347932024753340_00-055-167.png            1   

   Cardiomegaly  Effusion  Infiltration  Mass  Nodule  Pneumonia  \
0             0         0             0     0       0          0   
1             0         0             0     0       0          0   
2             0         0             0     0       0          0   
3             0         0             0     0       0          0   
4             0         0             0     0 

In [1]:
import pandas as pd
from collections import Counter

# =========================
# CONFIG
# =========================
CSV_PATH = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"

# =========================
# LOAD CSV
# =========================
df = pd.read_csv(CSV_PATH)

# =========================
# FILTER AP ONLY
# =========================
df_ap = df[
    df["View Position"].astype(str).str.strip() == "AP"
].copy()

print("=" * 80)
print("จำนวนภาพ AP ทั้งหมด:", len(df_ap))
print("=" * 80)

# =========================
# EXTRACT LABELS
# =========================
all_labels = []

for labels in df_ap["Finding Labels"].dropna():

    # split multi-label
    split_labels = labels.split("|")

    for label in split_labels:

        # clean text
        label = label.strip()

        # skip empty
        if label == "":
            continue

        all_labels.append(label)

# =========================
# COUNT LABELS
# =========================
label_counter = Counter(all_labels)

# =========================
# SORT
# =========================
sorted_labels = sorted(
    label_counter.items(),
    key=lambda x: x[1],
    reverse=True
)

# =========================
# SHOW RESULT
# =========================
print("=" * 80)
print(f"จำนวนโรค/label ไม่ซ้ำทั้งหมด (AP ONLY): {len(sorted_labels)}")
print("=" * 80)

for i, (label, count) in enumerate(sorted_labels, 1):
    print(f"{i}. {label} : {count}")

จำนวนภาพ AP ทั้งหมด: 44810
จำนวนโรค/label ไม่ซ้ำทั้งหมด (AP ONLY): 15
1. No Finding : 21059
2. Infiltration : 10541
3. Effusion : 6728
4. Atelectasis : 5831
5. Consolidation : 3146
6. Mass : 2215
7. Nodule : 2154
8. Edema : 2027
9. Pneumothorax : 1895
10. Cardiomegaly : 1213
11. Emphysema : 1017
12. Pleural_Thickening : 967
13. Pneumonia : 801
14. Fibrosis : 278
15. Hernia : 35


In [1]:
import pandas as pd

# =========================
# CONFIG
# =========================
CSV_PATH = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"

# =========================
# LOAD CSV
# =========================
df = pd.read_csv(CSV_PATH)

# ลบช่องว่างหน้า-หลัง
df["View Position"] = df["View Position"].astype(str).str.strip()

# =========================
# COUNT VIEW POSITIONS
# =========================
view_counts = df["View Position"].value_counts()

print("=" * 50)
print("View Position Distribution")
print("=" * 50)

print(f"AP : {view_counts.get('AP', 0):,} images")
print(f"PA : {view_counts.get('PA', 0):,} images")

# ถ้ามี View อื่น ๆ ด้วย
print("\nAll View Positions:")
print(view_counts)

View Position Distribution
AP : 44,810 images
PA : 67,310 images

All View Positions:
View Position
PA    67310
AP    44810
Name: count, dtype: int64


In [2]:
print(df["View Position"].value_counts())

print(df["View Position"].value_counts(normalize=True) * 100)

View Position
PA    67310
AP    44810
Name: count, dtype: int64
View Position
PA    60.033892
AP    39.966108
Name: proportion, dtype: float64


In [2]:
import shutil
from pathlib import Path

import pandas as pd


# =========================================================
# CONFIG
# =========================================================

# ไฟล์ CSV เดิม
CSV_PATH = Path(
    r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"
)

# โฟลเดอร์หลักที่เก็บรูป NIH เดิม
# โค้ดจะค้นหาภายในโฟลเดอร์ย่อยทั้งหมดให้อัตโนมัติ
# เช่น images_001, images_002, images_003 เป็นต้น
IMAGE_ROOT = Path(
    r"D:\senior-project\ploy-senior-project\archive"
)

# โฟลเดอร์ใหม่สำหรับเก็บภาพ AP
OUTPUT_FOLDER = Path(
    r"D:\senior-project\ploy-senior-project\nih_AP_chest"
)

# ---------------------------------------------------------
# รอบแรกให้ใช้ True เพื่อตรวจสอบอย่างเดียว ยังไม่ Copy
# เมื่อผลตรวจสอบเรียบร้อยแล้ว เปลี่ยนเป็น False
# ---------------------------------------------------------
DRY_RUN = False

# หากปลายทางมีภาพอยู่แล้ว
# False = ข้าม ไม่เขียนทับ
# True  = เขียนทับภาพเดิม
OVERWRITE = False

# ประเภทไฟล์ภาพที่ต้องการค้นหา
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


# =========================================================
# VALIDATE CONFIG
# =========================================================

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"ไม่พบไฟล์ CSV:\n{CSV_PATH}"
    )

if not IMAGE_ROOT.exists():
    raise FileNotFoundError(
        f"ไม่พบโฟลเดอร์ภาพต้นฉบับ:\n{IMAGE_ROOT}"
    )

# ป้องกันการตั้ง OUTPUT_FOLDER เป็นโฟลเดอร์เดียวกับ IMAGE_ROOT
if OUTPUT_FOLDER.resolve() == IMAGE_ROOT.resolve():
    raise ValueError(
        "OUTPUT_FOLDER ต้องไม่ใช่โฟลเดอร์เดียวกับ IMAGE_ROOT"
    )


# =========================================================
# LOAD CSV
# =========================================================

print("=" * 80)
print("กำลังอ่านไฟล์ CSV")
print("=" * 80)

df = pd.read_csv(CSV_PATH)

required_columns = {
    "Image Index",
    "View Position"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"ไฟล์ CSV ไม่มีคอลัมน์ที่จำเป็น: {missing_columns}"
    )


# =========================================================
# CLEAN CSV DATA
# =========================================================

df["Image Index"] = (
    df["Image Index"]
    .astype(str)
    .str.strip()
)

df["View Position"] = (
    df["View Position"]
    .astype(str)
    .str.strip()
    .str.upper()
)


# =========================================================
# FILTER AP ONLY
# =========================================================

df_ap = df[
    df["View Position"] == "AP"
].copy()

ap_filenames = df_ap["Image Index"].tolist()

print(f"จำนวนแถวทั้งหมดใน CSV : {len(df):,}")
print(f"จำนวนภาพ AP ใน CSV    : {len(df_ap):,}")
print(f"จำนวนภาพ PA ใน CSV    : {(df['View Position'] == 'PA').sum():,}")


# =========================================================
# CHECK DUPLICATE IMAGE NAMES IN AP CSV
# =========================================================

duplicate_ap_names = (
    df_ap["Image Index"]
    .value_counts()
)

duplicate_ap_names = duplicate_ap_names[
    duplicate_ap_names > 1
]

print("=" * 80)
print("ตรวจสอบชื่อภาพ AP ซ้ำใน CSV")
print("=" * 80)

if duplicate_ap_names.empty:
    print("ไม่พบชื่อภาพ AP ซ้ำใน CSV")
else:
    print(
        f"พบชื่อภาพ AP ซ้ำใน CSV "
        f"{len(duplicate_ap_names):,} ชื่อ"
    )

    print(duplicate_ap_names.head(20))


# =========================================================
# BUILD IMAGE INDEX
# =========================================================

print("=" * 80)
print("กำลังค้นหาภาพต้นฉบับ")
print(f"IMAGE_ROOT: {IMAGE_ROOT}")
print("=" * 80)

image_index = {}
duplicate_source_names = {}

output_resolved = OUTPUT_FOLDER.resolve()

for file_path in IMAGE_ROOT.rglob("*"):

    if not file_path.is_file():
        continue

    if file_path.suffix.lower() not in IMAGE_EXTENSIONS:
        continue

    # ป้องกันไม่ให้โค้ดอ่านภาพจาก OUTPUT_FOLDER
    # กลับมาเป็นภาพต้นฉบับ เมื่อรันโค้ดซ้ำ
    try:
        file_path.resolve().relative_to(output_resolved)
        continue
    except ValueError:
        pass

    filename = file_path.name

    if filename in image_index:

        if filename not in duplicate_source_names:
            duplicate_source_names[filename] = [
                image_index[filename]
            ]

        duplicate_source_names[filename].append(file_path)

    else:
        image_index[filename] = file_path


print(f"พบชื่อไฟล์ภาพต้นฉบับไม่ซ้ำ : {len(image_index):,}")

if duplicate_source_names:
    print(
        f"คำเตือน: พบชื่อภาพซ้ำในโฟลเดอร์ต้นฉบับ "
        f"{len(duplicate_source_names):,} ชื่อ"
    )
else:
    print("ไม่พบชื่อภาพซ้ำในโฟลเดอร์ต้นฉบับ")


# =========================================================
# CHECK AP IMAGES
# =========================================================

print("=" * 80)
print("ตรวจสอบภาพ AP ก่อน Copy")
print("=" * 80)

found_images = []
missing_images = []
invalid_projection_rows = []

for image_name in ap_filenames:

    source_path = image_index.get(image_name)

    if source_path is None:
        missing_images.append(image_name)
    else:
        found_images.append(
            {
                "Image Index": image_name,
                "Source Path": str(source_path)
            }
        )


# ตรวจสอบเพิ่มเติมว่าไฟล์ AP ไม่มีค่า projection อื่นหลุดมา
invalid_projection_rows = df_ap[
    df_ap["View Position"] != "AP"
]

print(f"จำนวน AP ที่ระบุใน CSV        : {len(ap_filenames):,}")
print(f"จำนวนภาพ AP ที่ค้นพบจริง      : {len(found_images):,}")
print(f"จำนวนภาพ AP ที่หาไม่พบ        : {len(missing_images):,}")
print(f"จำนวนแถวที่ไม่ใช่ AP หลุดมา   : {len(invalid_projection_rows):,}")


# =========================================================
# SAVE CHECK REPORTS
# =========================================================

report_folder = OUTPUT_FOLDER.parent / "nih_AP_check_report"
report_folder.mkdir(parents=True, exist_ok=True)

found_report_path = report_folder / "found_AP_images.csv"
missing_report_path = report_folder / "missing_AP_images.csv"
duplicate_report_path = report_folder / "duplicate_source_images.csv"

pd.DataFrame(found_images).to_csv(
    found_report_path,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame({
    "Image Index": missing_images
}).to_csv(
    missing_report_path,
    index=False,
    encoding="utf-8-sig"
)

if duplicate_source_names:

    duplicate_rows = []

    for image_name, paths in duplicate_source_names.items():
        for path in paths:
            duplicate_rows.append({
                "Image Index": image_name,
                "Source Path": str(path)
            })

    pd.DataFrame(duplicate_rows).to_csv(
        duplicate_report_path,
        index=False,
        encoding="utf-8-sig"
    )


print("=" * 80)
print("ไฟล์รายงานการตรวจสอบ")
print("=" * 80)

print(f"รายการภาพที่พบ    : {found_report_path}")
print(f"รายการภาพที่ไม่พบ : {missing_report_path}")

if duplicate_source_names:
    print(f"รายการชื่อภาพซ้ำ   : {duplicate_report_path}")


# =========================================================
# STOP WHEN DRY RUN
# =========================================================

if DRY_RUN:

    print("\n" + "=" * 80)
    print("DRY RUN: ตรวจสอบอย่างเดียว ยังไม่มีการ Copy ภาพ")
    print("=" * 80)

    if len(missing_images) == 0 and len(found_images) == len(ap_filenames):
        print("ผลตรวจสอบผ่าน")
        print(f"พบภาพ AP ครบทั้งหมด {len(found_images):,} ภาพ")
        print("")
        print("ขั้นตอนถัดไป:")
        print("เปลี่ยน DRY_RUN = True")
        print("เป็น")
        print("DRY_RUN = False")
        print("แล้วรันโค้ดอีกครั้งเพื่อ Copy จริง")

    else:
        print("ผลตรวจสอบยังไม่ผ่าน")
        print(
            f"มีภาพ AP หาไม่พบจำนวน "
            f"{len(missing_images):,} ภาพ"
        )
        print("กรุณาตรวจสอบ IMAGE_ROOT และไฟล์ missing_AP_images.csv")

    raise SystemExit


# =========================================================
# SAFETY CHECK BEFORE REAL COPY
# =========================================================

if missing_images:
    raise RuntimeError(
        f"ยกเลิกการ Copy เพราะมีภาพ AP หาไม่พบ "
        f"{len(missing_images):,} ภาพ\n"
        f"ตรวจสอบไฟล์: {missing_report_path}"
    )

if len(found_images) != len(ap_filenames):
    raise RuntimeError(
        "ยกเลิกการ Copy เพราะจำนวนภาพที่พบไม่ตรงกับ CSV"
    )

if duplicate_source_names:
    raise RuntimeError(
        "ยกเลิกการ Copy เพราะพบชื่อภาพซ้ำในโฟลเดอร์ต้นฉบับ\n"
        f"ตรวจสอบไฟล์: {duplicate_report_path}"
    )


# =========================================================
# CREATE OUTPUT FOLDER
# =========================================================

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 80)
print("เริ่ม Copy ภาพ AP")
print(f"ปลายทาง: {OUTPUT_FOLDER}")
print("=" * 80)


# =========================================================
# COPY AP IMAGES
# =========================================================

copied_count = 0
skipped_count = 0
failed_count = 0
copy_errors = []

total_images = len(found_images)

for number, image_info in enumerate(found_images, start=1):

    image_name = image_info["Image Index"]
    source_path = Path(image_info["Source Path"])
    destination_path = OUTPUT_FOLDER / image_name

    try:

        if destination_path.exists() and not OVERWRITE:
            skipped_count += 1
        else:
            shutil.copy2(
                source_path,
                destination_path
            )
            copied_count += 1

    except Exception as error:

        failed_count += 1

        copy_errors.append({
            "Image Index": image_name,
            "Source Path": str(source_path),
            "Destination Path": str(destination_path),
            "Error": str(error)
        })

    if number % 1000 == 0 or number == total_images:
        print(
            f"ดำเนินการแล้ว "
            f"{number:,}/{total_images:,} ภาพ"
        )


# =========================================================
# VERIFY OUTPUT AFTER COPY
# =========================================================

print("=" * 80)
print("กำลังตรวจสอบโฟลเดอร์ปลายทางหลัง Copy")
print("=" * 80)

output_files = {
    file_path.name
    for file_path in OUTPUT_FOLDER.iterdir()
    if (
        file_path.is_file()
        and file_path.suffix.lower() in IMAGE_EXTENSIONS
    )
}

expected_files = set(ap_filenames)

missing_after_copy = sorted(
    expected_files - output_files
)

unexpected_files = sorted(
    output_files - expected_files
)


# =========================================================
# SAVE COPY ERROR REPORT
# =========================================================

if copy_errors:

    copy_error_path = (
        report_folder
        / "copy_errors.csv"
    )

    pd.DataFrame(copy_errors).to_csv(
        copy_error_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"ไฟล์ข้อผิดพลาดการ Copy: {copy_error_path}")


# =========================================================
# SAVE FINAL VERIFICATION REPORT
# =========================================================

final_report_path = (
    report_folder
    / "final_AP_copy_verification.csv"
)

final_report = pd.DataFrame({
    "Check": [
        "AP rows in CSV",
        "Expected unique AP filenames",
        "Files found before copy",
        "Files copied in this run",
        "Files skipped because already existed",
        "Copy failures",
        "Files currently in output folder",
        "Missing files after copy",
        "Unexpected files in output folder"
    ],
    "Count": [
        len(df_ap),
        len(expected_files),
        len(found_images),
        copied_count,
        skipped_count,
        failed_count,
        len(output_files),
        len(missing_after_copy),
        len(unexpected_files)
    ]
})

final_report.to_csv(
    final_report_path,
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# FINAL RESULT
# =========================================================

print("\n" + "=" * 80)
print("สรุปผลการ Copy")
print("=" * 80)

print(f"จำนวนภาพ AP ตาม CSV             : {len(df_ap):,}")
print(f"จำนวนชื่อภาพ AP ไม่ซ้ำ          : {len(expected_files):,}")
print(f"Copy สำเร็จในรอบนี้             : {copied_count:,}")
print(f"ข้ามเพราะมีไฟล์อยู่แล้ว         : {skipped_count:,}")
print(f"Copy ไม่สำเร็จ                  : {failed_count:,}")
print(f"จำนวนภาพใน nih_AP_chest ปัจจุบัน: {len(output_files):,}")
print(f"ภาพที่ขาดหลัง Copy              : {len(missing_after_copy):,}")
print(f"ภาพที่ไม่ควรอยู่ในโฟลเดอร์      : {len(unexpected_files):,}")

print(f"\nโฟลเดอร์ภาพ AP:")
print(OUTPUT_FOLDER)

print(f"\nรายงานตรวจสอบ:")
print(final_report_path)


if (
    len(missing_after_copy) == 0
    and len(unexpected_files) == 0
    and failed_count == 0
    and len(output_files) == len(expected_files)
):
    print("\n" + "=" * 80)
    print("ตรวจสอบผ่าน: ภาพ AP ถูก Copy ครบและถูกต้อง")
    print("=" * 80)

else:
    print("\n" + "=" * 80)
    print("ตรวจสอบไม่ผ่าน: กรุณาดูรายงานและข้อความด้านบน")
    print("=" * 80)

    if missing_after_copy:
        print("\nตัวอย่างภาพที่ยังขาด:")
        for image_name in missing_after_copy[:20]:
            print("-", image_name)

    if unexpected_files:
        print("\nตัวอย่างภาพที่ไม่ควรอยู่ในโฟลเดอร์:")
        for image_name in unexpected_files[:20]:
            print("-", image_name)

กำลังอ่านไฟล์ CSV
จำนวนแถวทั้งหมดใน CSV : 112,120
จำนวนภาพ AP ใน CSV    : 44,810
จำนวนภาพ PA ใน CSV    : 67,310
ตรวจสอบชื่อภาพ AP ซ้ำใน CSV
ไม่พบชื่อภาพ AP ซ้ำใน CSV
กำลังค้นหาภาพต้นฉบับ
IMAGE_ROOT: D:\senior-project\ploy-senior-project\archive
พบชื่อไฟล์ภาพต้นฉบับไม่ซ้ำ : 112,120
ไม่พบชื่อภาพซ้ำในโฟลเดอร์ต้นฉบับ
ตรวจสอบภาพ AP ก่อน Copy
จำนวน AP ที่ระบุใน CSV        : 44,810
จำนวนภาพ AP ที่ค้นพบจริง      : 44,810
จำนวนภาพ AP ที่หาไม่พบ        : 0
จำนวนแถวที่ไม่ใช่ AP หลุดมา   : 0
ไฟล์รายงานการตรวจสอบ
รายการภาพที่พบ    : D:\senior-project\ploy-senior-project\nih_AP_check_report\found_AP_images.csv
รายการภาพที่ไม่พบ : D:\senior-project\ploy-senior-project\nih_AP_check_report\missing_AP_images.csv
เริ่ม Copy ภาพ AP
ปลายทาง: D:\senior-project\ploy-senior-project\nih_AP_chest
ดำเนินการแล้ว 1,000/44,810 ภาพ
ดำเนินการแล้ว 2,000/44,810 ภาพ
ดำเนินการแล้ว 3,000/44,810 ภาพ
ดำเนินการแล้ว 4,000/44,810 ภาพ
ดำเนินการแล้ว 5,000/44,810 ภาพ
ดำเนินการแล้ว 6,000/44,810 ภาพ
ดำเนินการแล้ว 7,000/44,810 ภาพ
ด

สร้าง csv file

In [3]:
import pandas as pd
from pathlib import Path

# ==========================================================
# CONFIG
# ==========================================================

# CSV ต้นฉบับ
CSV_PATH = Path(
    r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"
)

# โฟลเดอร์ใหม่
OUTPUT_FOLDER = Path(
    r"D:\senior-project\ploy-senior-project\nih_AP_chest"
)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# ==========================================================
# LOAD CSV
# ==========================================================

print("=" * 80)
print("Loading CSV...")
print("=" * 80)

df = pd.read_csv(CSV_PATH)

# ==========================================================
# FILTER AP ONLY
# ==========================================================

df["View Position"] = (
    df["View Position"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df_ap = df[df["View Position"] == "AP"].copy()

# ==========================================================
# SAVE NEW CSV
# ==========================================================

OUTPUT_CSV = OUTPUT_FOLDER / "Data_Entry_2017_AP.csv"

df_ap.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

# ==========================================================
# SUMMARY
# ==========================================================

print("=" * 80)
print("CSV CREATED SUCCESSFULLY")
print("=" * 80)

print(f"Original rows : {len(df):,}")
print(f"AP rows       : {len(df_ap):,}")
print(f"PA removed    : {len(df)-len(df_ap):,}")

print("\nSaved to:")
print(OUTPUT_CSV)

print("\nColumns:")

for i, col in enumerate(df_ap.columns, 1):
    print(f"{i}. {col}")

print("=" * 80)

Loading CSV...
CSV CREATED SUCCESSFULLY
Original rows : 112,120
AP rows       : 44,810
PA removed    : 67,310

Saved to:
D:\senior-project\ploy-senior-project\nih_AP_chest\Data_Entry_2017_AP.csv

Columns:
1. Image Index
2. Finding Labels
3. Follow-up #
4. Patient ID
5. Patient Age
6. Patient Gender
7. View Position
8. OriginalImage[Width
9. Height]
10. OriginalImagePixelSpacing[x
11. y]
12. Unnamed: 11


In [1]:
import pandas as pd

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = r"D:\senior-project\ploy-senior-project\archive\Data_Entry_2017.csv"

# ============================================================
# LOAD CSV
# ============================================================
df = pd.read_csv(CSV_PATH)

print("=" * 80)
print("DATA QUALITY CHECK : NIH ChestX-ray14")
print("=" * 80)

print(f"จำนวนแถวทั้งหมด   : {len(df):,}")
print(f"จำนวนคอลัมน์      : {len(df.columns):,}")

print("\nชื่อคอลัมน์:")
for col in df.columns:
    print(f"- {col}")


# ============================================================
# 2.1 MISSING VALUES
# ============================================================
print("\n" + "=" * 80)
print("2.1 MISSING VALUES")
print("=" * 80)

missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing (%)": missing_percent
})

print(missing_report.to_string())

total_missing = df.isnull().sum().sum()

print("\nสรุป:")
print(f"จำนวน Missing Values ทั้งหมด : {total_missing:,}")

if total_missing == 0:
    print("✓ ไม่พบ Missing Values")
else:
    print("⚠ พบ Missing Values")


# ============================================================
# ตรวจค่าว่างที่อาจไม่ถูกนับเป็น NaN
# เช่น "", " ", "NA", "N/A", "null"
# ============================================================
print("\n" + "-" * 80)
print("ตรวจ Blank / Empty String")
print("-" * 80)

text_columns = df.select_dtypes(include=["object"]).columns

for col in text_columns:

    # แปลงเป็น string และตัดช่องว่าง
    cleaned = df[col].fillna("").astype(str).str.strip()

    blank_count = (cleaned == "").sum()

    print(f"{col:<35} : {blank_count:,}")


# ============================================================
# 2.2 DUPLICATE
# ============================================================
print("\n" + "=" * 80)
print("2.2 DUPLICATE CHECK")
print("=" * 80)

# ------------------------------------------------------------
# A. ตรวจแถวที่เหมือนกันทั้งหมด
# ------------------------------------------------------------
full_duplicates = df.duplicated().sum()

print(f"Duplicate rows (ทั้งแถวเหมือนกัน) : {full_duplicates:,}")


# ------------------------------------------------------------
# B. ตรวจ Image Index ซ้ำ
# ------------------------------------------------------------
duplicate_images = df["Image Index"].duplicated().sum()

print(f"Duplicate Image Index              : {duplicate_images:,}")

if duplicate_images > 0:

    print("\nตัวอย่าง Image Index ที่ซ้ำ:")

    duplicated_image_rows = df[
        df["Image Index"].duplicated(keep=False)
    ].sort_values("Image Index")

    print(
        duplicated_image_rows[
            ["Image Index", "Patient ID", "Finding Labels"]
        ].head(20).to_string(index=False)
    )

else:
    print("✓ ไม่พบ Image Index ซ้ำ")


# ------------------------------------------------------------
# C. Patient ID ซ้ำ
# ------------------------------------------------------------
# หมายเหตุ:
# Patient ID ซ้ำไม่ถือว่าเป็นข้อมูลผิด
# เพราะผู้ป่วยหนึ่งคนสามารถมีภาพ X-ray หลายภาพได้
# ------------------------------------------------------------

unique_patients = df["Patient ID"].nunique()

print(f"\nจำนวน Patient ทั้งหมด (Unique)    : {unique_patients:,}")

images_per_patient = df.groupby("Patient ID").size()

print(f"จำนวนภาพน้อยที่สุดต่อ Patient      : {images_per_patient.min():,}")
print(f"จำนวนภาพมากที่สุดต่อ Patient       : {images_per_patient.max():,}")
print(f"ค่าเฉลี่ยภาพต่อ Patient             : {images_per_patient.mean():.2f}")


# ============================================================
# 2.3 ABNORMAL / INVALID VALUES
# ============================================================
print("\n" + "=" * 80)
print("2.3 ABNORMAL / INVALID VALUES")
print("=" * 80)


# ============================================================
# A. PATIENT AGE
# ============================================================
print("\n[A] Patient Age")
print("-" * 80)

print(df["Patient Age"].describe())

# ตรวจอายุที่เป็นไปไม่ได้/น่าสงสัย
# ณ ขั้นนี้ยังไม่ลบ แค่รายงาน
abnormal_age = df[
    (df["Patient Age"] < 0) |
    (df["Patient Age"] > 120)
]

print(f"\nจำนวน Age < 0 หรือ > 120 : {len(abnormal_age):,}")

if len(abnormal_age) > 0:

    print("\nรายการอายุผิดปกติ:")

    print(
        abnormal_age[
            [
                "Image Index",
                "Patient ID",
                "Patient Age",
                "Patient Gender",
                "View Position"
            ]
        ].to_string(index=False)
    )

else:
    print("✓ ไม่พบอายุ < 0 หรือ > 120")


# ============================================================
# B. PATIENT GENDER
# ============================================================
print("\n[B] Patient Gender")
print("-" * 80)

gender_clean = df["Patient Gender"].astype(str).str.strip().str.upper()

print("ค่าที่พบ:")
print(gender_clean.value_counts(dropna=False))

valid_gender = ["M", "F"]

invalid_gender = df[
    ~gender_clean.isin(valid_gender)
]

print(f"\nจำนวน Gender ที่ไม่ใช่ M/F : {len(invalid_gender):,}")

if len(invalid_gender) == 0:
    print("✓ Gender ถูกต้องทั้งหมด")
else:
    print("\nค่าที่ผิดปกติ:")
    print(invalid_gender["Patient Gender"].value_counts(dropna=False))


# ============================================================
# C. VIEW POSITION
# ============================================================
print("\n[C] View Position")
print("-" * 80)

view_clean = df["View Position"].astype(str).str.strip().str.upper()

print("ค่าที่พบ:")
print(view_clean.value_counts(dropna=False))

valid_views = ["AP", "PA"]

invalid_view = df[
    ~view_clean.isin(valid_views)
]

print(f"\nจำนวน View Position ที่ไม่ใช่ AP/PA : {len(invalid_view):,}")

if len(invalid_view) == 0:
    print("✓ View Position ถูกต้องทั้งหมด")
else:
    print("\nค่าที่ผิดปกติ:")
    print(invalid_view["View Position"].value_counts(dropna=False))


# ============================================================
# D. FINDING LABELS
# ============================================================
print("\n[D] Finding Labels")
print("-" * 80)

valid_labels = {
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia",
    "No Finding"
}

# แยก label ที่คั่นด้วย |
all_labels = set()

for labels in df["Finding Labels"].dropna():

    for label in str(labels).split("|"):
        all_labels.add(label.strip())

print("Labels ที่พบใน CSV:")

for label in sorted(all_labels):
    print(f"- {label}")

unknown_labels = all_labels - valid_labels

print()

if len(unknown_labels) == 0:
    print("✓ ไม่พบ Finding Label ที่อยู่นอกเหนือรายการที่กำหนด")
else:
    print("⚠ พบ Finding Label ที่ไม่รู้จัก:")
    for label in sorted(unknown_labels):
        print(f"- {label}")


# ============================================================
# E. ORIGINAL IMAGE SIZE
# ============================================================
print("\n[E] Original Image Size")
print("-" * 80)

# NIH CSV บางเวอร์ชันอาจมีชื่อ column แตกต่างกันเล็กน้อย
width_columns = [
    col for col in df.columns
    if "width" in col.lower()
]

height_columns = [
    col for col in df.columns
    if "height" in col.lower()
]

print("Width columns :", width_columns)
print("Height columns:", height_columns)

for col in width_columns:

    invalid_width = df[
        pd.to_numeric(df[col], errors="coerce") <= 0
    ]

    print(
        f"{col}: ค่า <= 0 จำนวน {len(invalid_width):,}"
    )

for col in height_columns:

    invalid_height = df[
        pd.to_numeric(df[col], errors="coerce") <= 0
    ]

    print(
        f"{col}: ค่า <= 0 จำนวน {len(invalid_height):,}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("DATA QUALITY SUMMARY")
print("=" * 80)

print(f"Rows                     : {len(df):,}")
print(f"Unique Images            : {df['Image Index'].nunique():,}")
print(f"Unique Patients          : {df['Patient ID'].nunique():,}")

print("-" * 80)

print(f"Missing Values           : {total_missing:,}")
print(f"Duplicate Rows           : {full_duplicates:,}")
print(f"Duplicate Image Index    : {duplicate_images:,}")
print(f"Abnormal Age (<0, >120)  : {len(abnormal_age):,}")
print(f"Invalid Gender           : {len(invalid_gender):,}")
print(f"Invalid View Position    : {len(invalid_view):,}")
print(f"Unknown Finding Labels   : {len(unknown_labels):,}")

print("=" * 80)
print("ตรวจสอบเสร็จสิ้น — ยังไม่มีการแก้ไขหรือลบข้อมูล")
print("=" * 80)

DATA QUALITY CHECK : NIH ChestX-ray14
จำนวนแถวทั้งหมด   : 112,120
จำนวนคอลัมน์      : 12

ชื่อคอลัมน์:
- Image Index
- Finding Labels
- Follow-up #
- Patient ID
- Patient Age
- Patient Gender
- View Position
- OriginalImage[Width
- Height]
- OriginalImagePixelSpacing[x
- y]
- Unnamed: 11

2.1 MISSING VALUES
                             Missing Count  Missing (%)
Image Index                              0          0.0
Finding Labels                           0          0.0
Follow-up #                              0          0.0
Patient ID                               0          0.0
Patient Age                              0          0.0
Patient Gender                           0          0.0
View Position                            0          0.0
OriginalImage[Width                      0          0.0
Height]                                  0          0.0
OriginalImagePixelSpacing[x              0          0.0
y]                                       0          0.0
Unnamed: 11        